# Day2 모델링 (2) - 트리 모델 비교와 앙상블

`02_modeling.ipynb`와 같은 분할(`src/data.py`, SEED 42, stratify)을 쓴다. 모든 비교는 학습 80%의 5-Fold 교차검증이고 20% 홀드아웃은 열지 않는다.
교차검증 예측 확률(OOF)을 모아서 모델별 F1, 최적 임계값, 앙상블 효과를 본다.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from xgboost import XGBClassifier

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_all  # noqa: E402
from preprocess import build_pipeline  # noqa: E402

warnings.filterwarnings("ignore")
SEED = 42
TAB = ROOT / "reports/tables"
CACHE = ROOT / "data/cache"

d = load_all(ROOT, seed=SEED)
X_tr, y_tr, GROUPS = d.X_tr, d.y_tr, d.groups
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("학습", X_tr.shape, "| label 1 비율", round(y_tr.mean(), 3))

학습 (6679, 3290) | label 1 비율 0.544


## 1. 모델 비교 (수치4 + RDKit + 핑거프린트)

피처군은 `cont+rdkit+fp`로 고정한다. 전처리(중앙값 대체, 상수열 제거)는 fold 안에서 fit한다. 하이퍼파라미터는 기본값에 가까운 설정이고 튜닝은 뒤에서 한다.

In [2]:
THS = np.round(np.arange(0.30, 0.71, 0.01), 2)
FEATURES = ["cont", "rdkit", "fp"]

MODELS = {
    "HistGB": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31, colsample_bytree=0.3, subsample=0.8,
                               subsample_freq=1, random_state=SEED, n_jobs=3, verbose=-1),
    "XGBoost": XGBClassifier(n_estimators=600, learning_rate=0.03, max_depth=6, colsample_bytree=0.3, subsample=0.8,
                             tree_method="hist", n_jobs=3, random_state=SEED, eval_metric="logloss"),
    "CatBoost": CatBoostClassifier(iterations=600, learning_rate=0.05, depth=6, rsm=0.1, thread_count=3,
                                   random_seed=SEED, verbose=0),
    "RandomForest": RandomForestClassifier(n_estimators=500, n_jobs=3, random_state=SEED),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=500, n_jobs=3, random_state=SEED),
}

OOF, ROWS = {}, []


def evaluate(name, model, features=FEATURES, **kw):
    pipe = build_pipeline(model, GROUPS, features, **kw)
    proba = cross_val_predict(pipe, X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=5)[:, 1]
    OOF[name] = proba
    f1s = np.array([f1_score(y_tr, (proba >= t).astype(int)) for t in THS])
    ROWS.append({"모델": name, "AUC": roc_auc_score(y_tr, proba), "F1@0.5": f1_score(y_tr, (proba >= 0.5).astype(int)),
                 "최대 F1": f1s.max(), "최적 임계값": THS[f1s.argmax()]})
    print(f"{name:<14} AUC {ROWS[-1]['AUC']:.4f} | F1@0.5 {ROWS[-1]['F1@0.5']:.4f} | 최대 F1 {ROWS[-1]['최대 F1']:.4f} @ {ROWS[-1]['최적 임계값']:.2f}")


for name, model in MODELS.items():
    evaluate(name, model)

HistGB         AUC 0.8774 | F1@0.5 0.8199 | 최대 F1 0.8233 @ 0.39


LightGBM       AUC 0.8817 | F1@0.5 0.8229 | 최대 F1 0.8273 @ 0.33


XGBoost        AUC 0.8810 | F1@0.5 0.8215 | 최대 F1 0.8271 @ 0.42


CatBoost       AUC 0.8750 | F1@0.5 0.8152 | 최대 F1 0.8224 @ 0.41


RandomForest   AUC 0.8768 | F1@0.5 0.8186 | 최대 F1 0.8232 @ 0.45


ExtraTrees     AUC 0.8695 | F1@0.5 0.8144 | 최대 F1 0.8228 @ 0.43


In [3]:
res = pd.DataFrame(ROWS).sort_values("AUC", ascending=False).reset_index(drop=True)
res.round(4).to_csv(TAB / "modeling_cv_models.csv", index=False)
pd.to_pickle(OOF, CACHE / "oof_03.pkl")
res.round(4)

,모델,AUC,F1@0.5,최대 F1,최적 임계값
0,LightGBM,0.8817,0.8229,0.8273,0.33
1,XGBoost,0.8810,0.8215,0.8271,0.42
2,HistGB,0.8774,0.8199,0.8233,0.39
3,RandomForest,0.8768,0.8186,0.8232,0.45
4,CatBoost,0.8750,0.8152,0.8224,0.41
5,ExtraTrees,0.8695,0.8144,0.8228,0.43


## 2. 앙상블 (OOF 확률 평균)

확률을 평균하면 모델 간 오류가 상쇄될 수 있다. 상위 모델을 하나씩 추가하며 AUC와 최대 F1을 본다. 앙상블 구성은 같은 OOF로 골랐으므로 약간 낙관적이다.

In [4]:
order = res["모델"].tolist()
rows = []
for k in range(1, len(order) + 1):
    members = order[:k]
    proba = np.mean([OOF[m] for m in members], axis=0)
    f1s = np.array([f1_score(y_tr, (proba >= t).astype(int)) for t in THS])
    rows.append({"구성(AUC 상위 k개)": "+".join(members), "k": k, "AUC": roc_auc_score(y_tr, proba),
                 "F1@0.5": f1_score(y_tr, (proba >= 0.5).astype(int)), "최대 F1": f1s.max(), "최적 임계값": THS[f1s.argmax()]})
ens = pd.DataFrame(rows)
ens.round(4).to_csv(TAB / "modeling_cv_ensembles.csv", index=False)
ens.round(4)

,구성(AUC 상위 k개),k,AUC,F1@0.5,최대 F1,최적 임계값
0,LightGBM,1,0.8817,0.8229,0.8273,0.33
1,LightGBM+XGBoost,2,0.8826,0.8218,0.8269,0.41
2,LightGBM+XGBoost+HistGB,3,0.8823,0.8228,0.8272,0.41
3,LightGBM+XGBoost+HistGB+RandomForest,4,0.8835,0.8232,0.8279,0.43
4,LightGBM+XGBoost+HistGB+RandomForest+CatBoost,5,0.8831,0.8247,0.8274,0.42
5,LightGBM+XGBoost+HistGB+RandomForest+CatBoost+...,6,0.8842,0.8250,0.8285,0.43
